# FIAP Tech Challenge 3

## End-to-end corrected notebook

## 1. Imports

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
import joblib


## 2. Load Data

In [3]:

flights = pd.read_csv("../data/flights.csv")
airlines = pd.read_csv("../data/airlines.csv")
airports = pd.read_csv("../data/airports.csv")

flights.shape


C:\Users\cleit\AppData\Local\Temp\ipykernel_20284\3782084390.py:1: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  flights = pd.read_csv("../data/flights.csv")


(5819079, 31)

## 3. Feature Engineering

In [4]:

flights["DELAYED"] = (flights["ARRIVAL_DELAY"] > 15).astype(int)
flights["DEP_HOUR"] = flights["DEPARTURE_TIME"] // 100
flights["IS_WEEKEND"] = flights["DAY_OF_WEEK"].isin([6, 7]).astype(int)


## 4. Dataset Freezing for Modeling

In [5]:

flights_model = flights.sample(n=300_000, random_state=42)
flights_model.shape


(300000, 34)

## 5. Supervised Learning

In [6]:

selected_features = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "MONTH",
    "DAY_OF_WEEK",
    "DEP_HOUR",
    "DISTANCE",
    "IS_WEEKEND"
]

X = flights_model[selected_features]
y = flights_model["DELAYED"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

categorical_features = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
numerical_features = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE", "IS_WEEKEND"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)


### 5.1 Logistic Regression

In [7]:

X_train_lr = X_train.sample(n=100_000, random_state=42)
y_train_lr = y_train.loc[X_train_lr.index]

log_reg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=300, n_jobs=-1))
])

log_reg_pipeline.fit(X_train_lr, y_train_lr)

y_pred_lr = log_reg_pipeline.predict(X_test)
y_proba_lr = log_reg_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lr))
print("ROC AUC:", roc_auc_score(y_test, y_proba_lr))


TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['int', 'str']

### 5.2 Random Forest

In [ ]:

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_proba_rf))


## 6. Unsupervised Learning

In [ ]:

flights_cluster = flights_model.copy()

cluster_features = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "DISTANCE"]
X_cluster = flights_cluster[cluster_features]

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_cluster_scaled)

flights_cluster["CLUSTER"] = clusters

flights_cluster.groupby("CLUSTER")[cluster_features].mean()


## 7. Save Model

In [ ]:

joblib.dump(rf_pipeline, "models/random_forest_delay_model.pkl")
